# Analisis de tiempos de post-procesamiento

Lee `log_postprocesos.txt` (log conciso, con duraciones exactas por etapa) y
`log_postprocesos_full.txt` (log completo, con una marca `===== experimento | etapa | timestamp =====`
al inicio de cada subproceso) para responder: **cuanto se demora cada etapa,
y cuanto falta para terminar lo pendiente**.

Las dos fuentes se parsean por separado y se comparan entre si como chequeo
cruzado: el log conciso ya trae la duracion exacta (`DONE | XXXs`); el log
completo se usa para aproximar duraciones via la diferencia de tiempo entre
marcas consecutivas (util si el log conciso llegara a faltar o cortarse).

In [ ]:
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## 0. Configuracion

In [ ]:
CONCISE_LOG = Path("log_postprocesos.txt")
FULL_LOG = Path("log_postprocesos_full.txt")

print(f"Concise log: {CONCISE_LOG}  (existe={CONCISE_LOG.exists()})")
print(f"Full log   : {FULL_LOG}  (existe={FULL_LOG.exists()})")

## 1. Parsear el log conciso (`log_postprocesos.txt`) -- fuente principal

Formato de linea:
```
[2026-08-05 04:06:47] axis_v05_hdbscan_ms2         | estimate_symmetry      | DONE    |   159.6s
```
`DONE` trae la duracion exacta ya calculada por `run_all_postprocessing.py`
(`time.time() - t0`), no hace falta aproximarla.

In [ ]:
CONCISE_RE = re.compile(
    r"^\[(?P<ts>[\d-]+ [\d:]+)\]\s+(?P<exp>\S+)\s*\|\s*(?P<stage>[^|]+?)\s*\|\s*"
    r"(?P<status>START|DONE|SKIP|FAIL|CLEAN|ABORTED)\s*(?:\|\s*(?P<dur>[\d.]+)s)?"
)


def parse_concise(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = CONCISE_RE.match(line)
            if not m:
                continue
            ts = datetime.strptime(m.group("ts"), "%Y-%m-%d %H:%M:%S")
            dur = float(m.group("dur")) if m.group("dur") else None
            rows.append((ts, m.group("exp"), m.group("stage").strip(), m.group("status"), dur))
    return pd.DataFrame(rows, columns=["timestamp", "experiment", "stage", "status", "duration_s"])


df_concise = parse_concise(CONCISE_LOG) if CONCISE_LOG.exists() else pd.DataFrame()
print(f"Lineas parseadas: {len(df_concise)}")
if len(df_concise):
    print(df_concise["status"].value_counts().to_string())

## 2. Parsear el log completo (`log_postprocesos_full.txt`) -- chequeo cruzado

Cada linea `===== experimento | etapa | timestamp =====` marca el INICIO de un
subproceso. La duracion se aproxima como la diferencia entre esta marca y la
siguiente (asume que las etapas corren secuencialmente, que es como las corre
`run_all_postprocessing.py`).

In [ ]:
MARKER_RE = re.compile(r"^===== (?P<exp>.+?) \| (?P<stage>.+?) \| (?P<ts>[\d-]+ [\d:.]+) =====")


def parse_full_markers(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = MARKER_RE.match(line)
            if not m:
                continue
            ts = datetime.strptime(m.group("ts"), "%Y-%m-%d %H:%M:%S.%f")
            rows.append((ts, m.group("exp"), m.group("stage").strip()))
    df = pd.DataFrame(rows, columns=["timestamp", "experiment", "stage"]).sort_values("timestamp").reset_index(drop=True)
    df["duration_s"] = (df["timestamp"].shift(-1) - df["timestamp"]).dt.total_seconds()
    return df


df_markers = parse_full_markers(FULL_LOG) if FULL_LOG.exists() else pd.DataFrame()
print(f"Marcas parseadas: {len(df_markers)}")
display(df_markers.head(10))

## 3. Duracion por tipo de etapa

Fuente principal: log conciso (`DONE` con duracion exacta). Se muestra tambien
la version aproximada del log completo al lado, para confirmar que coinciden.

In [ ]:
done = df_concise[(df_concise.status == "DONE") & df_concise.duration_s.notna()].copy()

by_stage_concise = (
    done.groupby("stage")["duration_s"]
    .agg(n="count", media="mean", mediana="median", min="min", max="max", total="sum")
    .sort_values("total", ascending=False)
)
print("=== Duracion por etapa (log conciso, exacta) ===")
display(by_stage_concise)

by_stage_markers = (
    df_markers.dropna(subset=["duration_s"])
    .groupby("stage")["duration_s"]
    .agg(n="count", media="mean", mediana="median", min="min", max="max")
    .sort_values("media", ascending=False)
)
print("\n=== Duracion por etapa (log completo, aproximada via delta de marcas) ===")
display(by_stage_markers)

## 4. Duracion total por experimento (todas sus etapas sumadas)

In [ ]:
by_experiment = (
    done.groupby("experiment")["duration_s"]
    .agg(n_etapas="count", total_s="sum")
    .assign(total_min=lambda d: d["total_s"] / 60)
    .sort_values("total_s", ascending=False)
)
display(by_experiment.head(20))

## 5. Fallos y abortos (para cruzar con las tablas de estado del servidor)

In [ ]:
problems = df_concise[df_concise.status.isin(["FAIL", "ABORTED"])]
print(f"Total FAIL/ABORTED: {len(problems)}")
display(problems[["timestamp", "experiment", "stage", "status"]])

## 6. Tiempo total ya invertido

In [ ]:
total_s = done["duration_s"].sum()
print(f"Tiempo total acumulado en etapas con DONE: {total_s/3600:.2f} h ({total_s/60:.1f} min)")

if len(df_concise):
    span = df_concise["timestamp"].max() - df_concise["timestamp"].min()
    print(f"Rango cubierto por el log: {df_concise['timestamp'].min()}  ->  {df_concise['timestamp'].max()}  ({span})")

## 7. Proyeccion: cuanto falta

Costo promedio observado por variante, segun de que parte tenga que arrancar:

- **Variante nueva desde cero** (`map_to_3d` + `estimate_symmetry` + 4x`evaluate` + `compare_results`): la mas cara, domina `map_to_3d`.
- **Variante de clustering/hdbscan** (reusa `mapped_points_3d` ya existente -- solo `estimate_symmetry` + 4x`evaluate` + `compare_results`).
- **Variante de patch-size** (necesita su propio `map_to_3d` con `--patch-size` -- mismo costo que una variante nueva).

Ajusta `n_pendientes_*` con la cantidad de variantes que te falten (por
ejemplo, tomando el conteo de "SIN CORRER" de la tabla de estado del
servidor) para estimar el tiempo restante.

In [ ]:
avg = by_stage_concise["media"]

cost_map3d = avg.get("map_to_3d", 0)
cost_estimate = avg.get("estimate_symmetry", 0)
cost_eval_total = sum(avg.get(f"evaluate[{m}]", 0) for m in ["svd", "ransac_svd", "svd_sde", "ransac_svd_sde"])
cost_compare = avg.get("compare_results", 0)

cost_nueva = cost_map3d + cost_estimate + cost_eval_total + cost_compare
cost_clustering = cost_estimate + cost_eval_total + cost_compare
cost_patch = cost_map3d + cost_estimate + cost_eval_total + cost_compare

print("Costo promedio observado por tipo de variante:")
print(f"  Nueva desde cero (baseline)      : {cost_nueva/60:6.1f} min  (map_to_3d={cost_map3d/60:.1f} + estimate={cost_estimate/60:.1f} + 4xevaluate={cost_eval_total/60:.1f} + compare={cost_compare/60:.1f})")
print(f"  Clustering / hdbscan (sin map_to_3d): {cost_clustering/60:6.1f} min")
print(f"  Patch-size (con map_to_3d nuevo)  : {cost_patch/60:6.1f} min")

# --- editar estos numeros con lo que falte segun la tabla de estado del servidor ---
n_pendientes_nueva = 0
n_pendientes_clustering = 0
n_pendientes_patch = 0

total_pendiente_s = (
    n_pendientes_nueva * cost_nueva
    + n_pendientes_clustering * cost_clustering
    + n_pendientes_patch * cost_patch
)
print(f"\nCon n_pendientes_nueva={n_pendientes_nueva}, n_pendientes_clustering={n_pendientes_clustering}, n_pendientes_patch={n_pendientes_patch}:")
print(f"  Tiempo estimado restante: {total_pendiente_s/3600:.2f} h ({total_pendiente_s/60:.1f} min)")

## 8. Escenario: repetir TODO el post-procesamiento desde cero

28 experimentos base (24 Flow A + 4 Flow B/C) x 7 variantes cada uno
(1 baseline + 1 cluster + 3 hdbscan_ms{2,3,5} + 2 patch{3,5}). Estimado
secuencial (una sola corrida, sin paralelizar) con los costos promedio
observados arriba.

In [ ]:
N_BASE_EXPERIMENTS = 28   # 24 Flow A + 4 Flow B/C (axis_v05_1 y plane_v04_1)

n_baseline_total = N_BASE_EXPERIMENTS * 1        # 1 baseline por experimento
n_clustering_total = N_BASE_EXPERIMENTS * 4      # cluster + hdbscan_ms2/3/5
n_patch_total = N_BASE_EXPERIMENTS * 2           # p3 + p5

total_desde_cero_s = (
    n_baseline_total * cost_nueva
    + n_clustering_total * cost_clustering
    + n_patch_total * cost_patch
)

print(f"Variantes 'nueva' (baseline)   : {n_baseline_total:>4}  x {cost_nueva/60:5.1f} min = {n_baseline_total*cost_nueva/3600:6.2f} h")
print(f"Variantes clustering/hdbscan   : {n_clustering_total:>4}  x {cost_clustering/60:5.1f} min = {n_clustering_total*cost_clustering/3600:6.2f} h")
print(f"Variantes patch-size           : {n_patch_total:>4}  x {cost_patch/60:5.1f} min = {n_patch_total*cost_patch/3600:6.2f} h")
print(f"\nTOTAL para repetir todo desde cero (secuencial): {total_desde_cero_s/3600:.2f} h  (~{total_desde_cero_s/3600/24:.2f} dias corriendo sin parar)")